# Phase 2 · PubMed search and abstracts

NCBI E-utilities uses the email from `NCBI_EMAIL` and optional `NCBI_API_KEY`. Searches and fetched records are cached locally as append-only JSONL; successful work is reused after interruption. Set small limits for a smoke run and clear them only for a full acquisition.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.pipelines.phase2_pubmed import run_phase2

CONFIG = load_yaml("configs/default.yaml")
print("NCBI_API_KEY set:", bool(os.environ.get("NCBI_API_KEY")))
MAX_QUERY_CALLS = 5
MAX_FETCH_BATCHES = 10
print("NCBI_EMAIL set:", bool(os.environ.get("NCBI_EMAIL")))
species = pd.read_parquet(Path(CONFIG["paths"]["taxonomy_release"]) / "species.parquet")
aliases = pd.read_parquet(Path(CONFIG["paths"]["taxonomy_release"]) / "aliases.parquet")
print(f"Species: {len(species):,}; aliases: {len(aliases):,}; workers: {CONFIG['pubmed']['max_workers']}")
print("Cache path:", CONFIG["paths"]["pubmed_cache"])

Before full acquisition, set `MAX_QUERY_CALLS = None` and `MAX_FETCH_BATCHES = None`. Request pacing defaults to 0.34 seconds without `NCBI_API_KEY` and 0.11 seconds with one; `NCBI_EMAIL` identifies the tool contact but does not increase the rate by itself.

In [ ]:
# Bounded notebook call. This notebook-level cap applies to newly searched organisms; all fetched PMIDs are cached by distinct PMID.
MATCHES, ABSTRACTS = run_phase2(max_calls=MAX_QUERY_CALLS, max_fetch_batches=MAX_FETCH_BATCHES)
print(f"Stored {len(MATCHES):,} species–PMID rows and {len(ABSTRACTS):,} unique abstract rows.")